# Quantum Kernel Classifier (Qiskit)

Step-by-step walkthrough: feature map → quantum kernel → SVM → comparison with a classical RBF SVM.

Run from the project folder with the `.venv` kernel selected.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import qkernel as qk
%matplotlib inline

## 1. Data
Quantum-native *ad hoc* dataset (2 features → 2 qubits), stratified split, features scaled to the feature map's natural range.

In [ ]:
X, y = qk.load_dataset('adhoc', n_samples=200, seed=0)
Xtr, Xte, ytr, yte = qk.prepare_split(X, y, seed=0, scale_range=qk.base_range('adhoc'))
print(Xtr.shape, Xte.shape)
plt.scatter(*Xtr.T, c=ytr, cmap='bwr', edgecolor='k'); plt.title('Training data'); plt.show()

## 2. Feature map
ZZ feature map, 2 repetitions, linear entanglement.

In [ ]:
fmap = qk.build_feature_map('zz', n_qubits=2, reps=2, entanglement='linear')
fmap.decompose().draw('mpl', fold=40)

## 3. Quantum kernel + SVM
`K[i, j] = |<phi(x_i)|phi(x_j)>|^2`, then an SVM on the precomputed Gram matrix.

In [ ]:
K_train = qk.fidelity_kernel(fmap, Xtr)
K_test = qk.fidelity_kernel(fmap, Xte, Xtr)
clf = qk.PrecomputedKernelSVM(seed=0).fit(K_train, ytr)
acc_q = accuracy_score(yte, clf.predict(K_test))
print(f'Quantum-kernel SVM accuracy: {acc_q:.3f}  (C={clf.C_}, KTA={qk.kernel_target_alignment(K_train, ytr):.3f})')
plt.imshow(K_train, cmap='viridis'); plt.colorbar(); plt.title('Quantum kernel matrix (train)'); plt.show()

## 4. Classical baseline

In [ ]:
sc = StandardScaler().fit(Xtr)
acc_c = qk.evaluate_classical('rbf', sc.transform(Xtr), sc.transform(Xte), ytr, yte)['acc']
print(f'Classical RBF SVM accuracy: {acc_c:.3f}')

## 5. Try it yourself
Change the dataset (`moons`, `circles`, `breast_cancer`), the feature map (`'z'`, `'xyz'`), `reps`, or the bandwidth (`qk.evaluate_quantum_tuned`). Full study: `python run_experiments.py`.